In [ ]:
import torch
import torch.nn as nn

<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

**📜 Concept Overview**  
Think of Multi-Head Attention as a team of “spotlights” scanning a sentence from different angles.  
- One head might focus on the **word right before**,  
- another on **keywords further away**,  
- and another on **overall sentence structure**.  

By running several of these spotlights in parallel and then combining their findings, the model builds a richer understanding of how words relate to each other — both near and far.

---

**⚙️ What This Code Does**  
1. **Setup & Dimensions:**  
   - Splits the model’s output size (`d_out`) evenly across `num_heads`.  
   - This means each head processes a smaller slice of the data (`head_dim`), making the work lighter and more specialized.  

2. **Project Inputs into Q, K, V:**  
   - Creates three different “views” of the input sequence:  
     - **Queries**: What I’m looking for.  
     - **Keys**: What I offer for matching.  
     - **Values**: What information I carry if I’m matched.  

3. **Split & Rearrange:**  
   - Reshapes the projections so each head works independently.  

4. **Attention Calculation:**  
   - Compares each query to all keys (`queries @ keysᵀ`) to find relevance scores.  
   - Applies a **causal mask** so the model can’t peek at future tokens.  
   - Scales the scores to stabilize training and uses **softmax** to turn them into probabilities.

5. **Weighted Information Gathering:**  
   - Uses the attention probabilities to pull in values from the most relevant tokens.  

6. **Combine & Output:**  
   - Merges all heads back together.  
   - Optionally applies a final linear layer to mix the information.

---

**💡 Why It Matters**  
In GPT, this is where raw token embeddings become **context-aware representations**.  
Instead of treating each word in isolation, Multi-Head Attention lets the model see **who’s talking to whom** in the sentence — capturing both **local detail** and **global meaning** in one step.

</div>


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

**📜 Concept Overview**  
This section adds the “fine-tuning” components that help GPT train stably and process information efficiently:  
- **LayerNorm:** Ensures each token’s features have a balanced scale and offset before further processing.  
- **GELU:** A smooth, non-linear activation that decides which signals to pass forward.  
- **FeedForward Network:** A mini neural network that transforms each token’s representation independently.

---

**⚙️ What This Code Does**  

1. **LayerNorm (`LayerNorm`)**  
   - Calculates the **mean** and **variance** for each token’s embedding vector.  
   - Normalizes values so features are on a similar scale.  
   - Learns two parameters:  
     - **Scale (`γ`)**: stretches or shrinks normalized values.  
     - **Shift (`β`)**: offsets the values after scaling.  
   - Adds `eps` to avoid division-by-zero errors.

2. **GELU Activation (`GELU`)**  
   - Instead of abruptly zeroing negative values (like ReLU), GELU smoothly reduces them.  
   - This is like having a “soft gate” — small signals are dampened, large signals pass through.  
   - Mathematically approximated with a **tanh** function for efficiency.

3. **FeedForward Network (`FeedForward`)**  
   - Two linear layers with a GELU in between:  
     1. **Expansion:** Increases the embedding dimension 4× to give the model more representational power.  
     2. **GELU Activation:** Adds non-linearity so the model can learn complex patterns.  
     3. **Contraction:** Reduces the dimension back to the original size.  
   - Operates **position-wise**, meaning it processes each token independently after attention.

---

**💡 Why It Matters**  
While attention lets tokens talk to each other, the **FeedForward + LayerNorm** block helps each token **refine its own understanding** before the next attention step.  
This keeps learning stable and ensures the model builds richer, more nuanced representations layer by layer.

</div>


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), ## Expansion
            GELU(), ## Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), ## Contraction
        )

    def forward(self, x):
        return self.layers(x)

<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

**📜 Concept Overview**  
A Transformer Block is like a **self-contained thinking unit** of GPT.  
It combines two main skills:  
1. **Multi-Head Attention:** Let tokens talk to each other and share information.  
2. **FeedForward Network:** Let each token refine and transform its own features.  

Both are wrapped in **Layer Normalization** and **Residual Connections** to keep learning stable and efficient.

---

**⚙️ What This Code Does**  

1. **Preparation:**  
   - Creates one **Multi-Head Attention** layer (`self.att`).  
   - Creates one **FeedForward** layer (`self.ff`).  
   - Adds **two LayerNorms**:  
     - `norm1` for attention inputs.  
     - `norm2` for feedforward inputs.  
   - Adds **dropout** (`drop_shortcut`) to prevent overfitting.

2. **Forward Pass:**  
   - **Step 1 – Attention + Residual:**  
     - Normalize input with `norm1`.  
     - Apply attention to exchange information between tokens.  
     - Add the result back to the original input (**residual shortcut**).  
     - Apply dropout to the shortcut path.  
   - **Step 2 – FeedForward + Residual:**  
     - Normalize again with `norm2`.  
     - Pass through the feedforward network to refine token representations.  
     - Add back to the intermediate result (**second residual**).  
     - Apply dropout.

---

**💡 Why It Matters**  
Stacking these blocks creates the **deep reasoning ability** of GPT.  
- **Attention** handles *relationships between tokens*.  
- **FeedForward** handles *individual token transformation*.  
- **Residuals + LayerNorm** keep gradients stable and allow deeper stacking without the network “forgetting” important features.

Think of each block as **a round of group discussion (attention)** followed by **personal reflection (feedforward)** — repeated many times for richer understanding.

</div>


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        # Attention + Residual
        x = x + self.drop_shortcut(self.att(self.norm1(x)))

        # FeedForward + Residual
        x = x + self.drop_shortcut(self.ff(self.norm2(x)))

        return x

<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

**📜 Concept Overview**  
This is the **full GPT architecture** assembled.  
It takes raw token indices, turns them into rich vector representations, processes them through multiple Transformer blocks, and finally predicts the next token for each position in the sequence.  
You can think of it as a **pipeline**:
> **Input IDs → Embeddings → Transformers → Prediction Layer → Output Probabilities**

---

**⚙️ What This Code Does**  

1. **Embedding Layers**  
   - `tok_emb`: Learns a unique vector for each token in the vocabulary (word meaning).  
   - `pos_emb`: Learns a vector for each position in the sequence (word order).  
   - Added together to give the model **both** semantic meaning and position information.

2. **Dropout on Embeddings**  
   - Randomly zeros out some embedding values during training to prevent overfitting.

3. **Transformer Blocks**  
   - A stack of `n_layers` identical **TransformerBlock** modules.  
   - Each block refines the token representations by mixing global context (attention) and local transformations (feedforward).

4. **Final Layer Normalization**  
   - Normalizes the output of the last Transformer block for stability.

5. **Output Head**  
   - A linear layer mapping each token vector to a **vocabulary-sized logits vector**.  
   - This gives unnormalized scores for **what the next token could be**.

---

**💡 Why It Matters**  
This class is the **heart of GPT**:  
- **Embeddings** turn discrete tokens into trainable continuous vectors.  
- **Transformer stack** builds context-aware representations.  
- **Output head** converts those representations back into token predictions.  

When trained, this pipeline learns to predict the next token so well that it can generate coherent text — one token at a time.

</div>


In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference

<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

**📜 Concept Overview**

This function generates text step-by-step using a trained GPT model. It starts with an initial sequence of token IDs and repeatedly predicts the next token until the desired length is reached.

---

**⚙️ What This Code Does**
1. **Context Cropping** – Ensures the input sequence never exceeds the model's maximum supported context size by keeping only the most recent tokens.
2. **Prediction Step** – Feeds the cropped context into the model to obtain logits (raw prediction scores for each token in the vocabulary).
3. **Next Token Selection** –  
   - Focuses on the last time step’s logits.  
   - Applies softmax to convert logits into probabilities.  
   - Chooses the token with the highest probability (greedy decoding).
4. **Sequence Extension** – Appends the chosen token to the input sequence and repeats until the target length is reached.

---

**💡 Why It Matters**

This is the **inference loop** that powers text generation in GPT-like models. By iteratively feeding the model’s own predictions back into itself, it can produce coherent sequences that extend beyond the original prompt.

</div>


In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context

    ###Input batch:
 ###tensor([[6109, 3626, 6100,  345],
        ##[6109, 1110, 6622,  257]])
    
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond) ### batch, n_tokens, vocab_size
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [ ]:
!pip install tiktoken


<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

 **Concept Overview**
 
This block connects the model’s text generation logic to actual readable text. It handles both the conversion of natural language into model-friendly token IDs and the decoding of generated tokens back into human-readable output.

---

 **What This Code Does**
1. **Tokenization Functions**  
   - `text_to_token_ids` → Converts a text prompt into numerical token IDs using the GPT-2 tokenizer. Adds a batch dimension for model compatibility.  
   - `token_ids_to_text` → Converts token IDs back into plain text by removing the batch dimension and decoding.
   
2. **Tokenizer Setup**  
   - Uses the `tiktoken` library with GPT-2’s vocabulary to ensure tokens align with the model’s training data.

3. **Text Generation Pipeline**  
   - Defines a starting prompt (`start_context`).  
   - Tokenizes it and feeds it into `generate_text_simple`.  
   - Generates `max_new_tokens=10` tokens while respecting the model’s `context_size`.

4. **Result Display**  
   - Decodes the generated token IDs and prints the final output text.

---

 **Why It Matters**

Language models work on numbers, not words. This step is the **bridge** between human-readable text and the numerical world the GPT operates in, enabling seamless input and output for the generation process.

</div>


In [ ]:
import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Part 2

In [ ]:
import os
import urllib.request

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [ ]:
print(text_data[:99])

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

**Concept Overview**

This code prepares raw text data into training-ready batches for GPT. It chunks the text into fixed-length token sequences and pairs each sequence with its shifted target sequence for next-token prediction.

---

**What This Code Does**
1. **`GPTDatasetV1` Class**  
   - **Tokenization** – Encodes the entire input text into token IDs using GPT-2’s tokenizer.  
   - **Sliding Window Chunking** – Creates overlapping chunks of length `max_length` using a `stride` value to control overlap.  
     - **`input_chunk`** → Tokens the model sees.  
     - **`target_chunk`** → Same tokens shifted by one position (the "next token" labels).  
   - Stores all `input_ids` and `target_ids` as PyTorch tensors.

2. **Dataloader Creation (`create_dataloader_v1`)**  
   - Initializes the tokenizer.  
   - Creates the dataset from the input text.  
   - Wraps it in a PyTorch `DataLoader` for efficient batching, shuffling, and parallel loading.

---

**Why It Matters**

Training GPT requires the model to **predict the next token** for every position in the input. This dataset structure ensures:  
- Fixed-length inputs for efficient GPU training.  
- Overlapping sequences so no context is wasted.  
- Batches ready for parallel processing in PyTorch.

</div>


In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

**Concept Overview**

This block splits the dataset into training and validation sets, then creates dataloaders for each. Training data is shuffled for learning, while validation data is kept in order for consistent evaluation.

---

**What This Code Does**

1. **Dataset Split**
   - `train_ratio = 0.90` → 90% of the text is used for training, 10% for validation.
   - Splits `text_data` at `split_idx` into `train_data` and `val_data`.

2. **Reproducibility**
   - Sets a fixed random seed (`torch.manual_seed(123)`) to ensure consistent shuffling across runs.

3. **Training Dataloader**
   - Uses `create_dataloader_v1` with:
     - `batch_size=2` for small, manageable batches.
     - `stride=context_length` to avoid overlap in sequences.
     - `shuffle=True` so each epoch sees data in a different order.
     - `drop_last=True` to keep batch sizes consistent.

4. **Validation Dataloader**
   - Similar to training but:
     - `shuffle=False` → maintains sequence order for evaluation.
     - `drop_last=False` → uses all remaining data, even if the batch is smaller.

---

**Why It Matters**

Separating training and validation ensures that:
- **Training**: The model learns patterns from shuffled, diverse batches.
- **Validation**: Performance is measured on unseen, ordered data for an unbiased accuracy check.  
This split is a **core principle in machine learning** for tracking overfitting and generalization.

</div>


In [ ]:
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Sanity check

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

print(len(train_loader))
print(len(val_loader))


In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel() # Returns the total number of elements (or tokens) in the input_batch.
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0: 
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

In [ ]:
# Note:
# Uncomment the following code to calculate the execution time
import time
start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# Uncomment the following code to show the execution time
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
import numpy as np
print(np.__version__)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import pandas as pd
import numpy as np


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

In [ ]:
model.to("cpu")
model.eval()

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Loading and Saving Model Weights in PyTorch (Usage of GPT 2 124M Parameter Model)

In [ ]:
model = GPTModel(GPT_CONFIG_124M)
torch.save(model.state_dict(), "model.pth")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    }, 
    "model_and_optimizer.pth"
)

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth")
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

In [ ]:
pip install tqdm>=4.66

In [ ]:
import tensorflow as tf
import tqdm

print("TensorFlow version:", tf.__version__)
print("tqdm version:", tqdm.__version__)

In [ ]:
from gpt_download3 import download_and_load_gpt2

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
print("Settings:", settings)
print("Parameter dictionary keys:", params.keys())

In [ ]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

In [ ]:
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])


In [ ]:
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})
gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, 
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, 
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, 
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, 
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, 
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, 
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, 
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, 
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, 
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, 
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])



In [ ]:
load_weights_into_gpt(gpt, params)
gpt.to(device);

<div style="background-color:#e8f5e9; color:black; padding:15px; border-radius:8px; font-size:15px; line-height:1.6;">

**Concept Overview**

These above blocks loads a pretrained GPT-2 (124M parameter) model, defines its configuration, and maps downloaded pretrained weights into the PyTorch implementation. This allows the model to generate text using knowledge learned during its original OpenAI training.

---

**What This Code Does**

1. **Version Checks**
   - Prints installed versions of `TensorFlow` and `tqdm` for reproducibility.

2. **Model Download**
   - Uses `download_and_load_gpt2` to fetch GPT-2 weights (`124M`) and configuration settings from local storage or the web.

3. **Configuration Setup**
   - Defines architecture parameters (`embedding dimension`, `layers`, `heads`) for multiple GPT-2 variants.
   - Updates the base configuration with the selected model’s settings plus custom values:
     - `context_length = 1024`
     - `qkv_bias = True` (enables query-key-value bias in attention).

4. **Weight Assignment Function**
   - `assign()` → Ensures weight tensors match expected shapes before loading into the model.

5. **Loading Pretrained Weights**
   - **Token & Positional Embeddings** → `wte` and `wpe`.
   - **Attention Layers** → Splits `c_attn` weights into Q, K, V matrices and loads them into the model’s attention projection layers.
   - **Feedforward Layers** → Loads fully connected (MLP) weights and biases.
   - **Layer Normalization** → Maps GPT-2’s layer norm parameters (`ln_1`, `ln_2`, and final norm).
   - **Output Head** → Shares weights with token embeddings for tied embedding-output representation.

---

**Why It Matters**

By loading pretrained weights:
- The model **retains language knowledge** from GPT-2’s large-scale training.
- It avoids the need for full training from scratch, enabling:
  - Fine-tuning on domain-specific data.
  - Immediate text generation with high-quality results.
  
This step effectively **bridges the architecture you’ve built with real-world learned parameters**, turning it from an empty shell into a functional GPT model.

</div>


In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("The Reason why the Lion is called king of the jungle is ", tokenizer).to(device),
    max_new_tokens=50,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Once there was farmer living in 1567 ", tokenizer).to(device),
    max_new_tokens=50,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))